In [ ]:
!git clone -b hemanth https://github.com/hemanthm1234/qrt-p2-automated-pipeline.git


### Importing Python Packages and Utility Functions

In [1]:
import sys
sys.path.append("/kaggle/working/qrt-p2-automated-pipeline")


In [2]:
import pandas as pd
import numpy as np

import datetime
import os, sys
import importlib

import utils
importlib.reload(utils)

from utils import plot_series, plot_series_with_names, plot_series_bar
from utils import plot_dataframe
from utils import get_universe_adjusted_series, scale_weights_to_one, scale_to_book_long_short
from utils import generate_portfolio, backtest_portfolio
from utils import match_implementations

import plotly.graph_objects as go
import matplotlib.pyplot as plt

### Data Loading and Structure

The dataset consists of three key files:


- **features.parquet**: A DataFrame containing **22 stock features** for each trading day from **2005 to 2025**, stored in a **columnar fashion**. For example, `features["macd"]` will return the `macd` of shape **(5068, 2167)**. All **22** features are guaranteed to have the same shape.

- **universe.parquet**: A DataFrame of the same shape as `features["macd"]`, containing `0` and `1` values, where `1` indicates that the stock is tradable on that day.

- **returns.parquet**: A DataFrame of shape **(3775, 2167)** containing **daily stock returns** from **2005 to 2019**. You will **never receive** the returns for the testing period i.e. from **2020 to 2025**.

### Data Organization:
- **Columns** represent stock identifiers, ranging from **1 to 2167** in increasing order.
  
- **Rows** represent trading days when the market was open.

In [3]:
# This directory can be used if you're working on a Kaggle Notebook inside the competition
# Change the directory as per your requirements if you're working somewhere else
data_dir = "/kaggle/input/notebooks/hem0627/qrt-p2-last-6months"

features = pd.read_parquet(os.path.join(data_dir, "features.parquet"))

universe = pd.read_parquet(os.path.join(data_dir, "universe_5m.parquet"))
 
returns = pd.read_parquet(os.path.join(data_dir, "returns.parquet"))

### Benchmark Strategy: Vectorized Portfolio Generation

In this step we will write a vectorized code to generate our strategy portfolio for all trading days at once without a `for` loop. For doing this we will use direct operations on feature dataframes. But, here's one caution. It is very easy to look into the future feature data when you're using dataframes to construct your portfolio at once.

Below is a vectorized implementation of the benchmark strategy [which you see on Kaggle]. Make sure to note the `alpha.shift(1)` operation in the code! This is to make sure you use only feature values upto the last trading day to construct today's portfolio weights.

You can change the code below to implement your own strategy. Since this vectorized implementation is faster, you can try various versions of your strategy and see their performances.

#### Inputs:
- `entire_features :pd.DataFrame`: Historical feature data (MultiIndex columns: Feature Names, Stock Identifiers).
- `universe: pd.DataFrame`: Binary DataFrame indicating tradable stocks per day.
- `start_date`, `end_date` :`str`: Backtest period in `'YYYY-MM-DD'` format.

#### Output:
- `portfolio(pd.DataFrame)`: Normalized portfolio weights for each stock per day in the specified date range.

In [4]:
def process_signal(signal, universe_boolean):
    signal = signal.shift(1)
    signal = signal.where(universe_boolean, np.nan)
    signal = signal.rank(axis=1, method="min", ascending=True)
    signal = signal.sub(signal.mean(axis=1), axis=0)
    signal = signal.div(signal.abs().sum(axis=1), axis=0)
    return signal.fillna(0)

In [5]:
processed_features = {}

def precompute_processed_features(end_date="2019-12-31"):
    universe_boolean = universe.loc[:end_date].astype(bool)
    features_ = features.loc[:end_date]

    # Extract unique feature names from the first level of the MultiIndex columns
    unique_features = features_.columns.get_level_values(0).unique()

    for fname in unique_features:
        # features_[fname] now correctly passes a 2D DataFrame (Dates x Stocks)
        processed_features[fname] = process_signal(features_[fname], universe_boolean)

# run once
precompute_processed_features()

feature_names = list(processed_features.keys())

feature_matrix = np.stack(
    [processed_features[f].values for f in feature_names],
    axis=-1
)

In [6]:
def evaluate_weights_fast(w, start_date="2005-01-03", end_date="2019-12-31"):
    w = np.asarray(w, dtype=float)
    if len(w) != len(feature_names):
        raise ValueError(f"Expected {len(feature_names)} weights, got {len(w)}")
        
    w = w / (np.sum(np.abs(w)) + 1e-8)
    
    scores = np.tensordot(feature_matrix, w, axes=([2], [0]))

    portfolio = pd.DataFrame(scores, 
                             index=processed_features[feature_names[0]].index,
                             columns=processed_features[feature_names[0]].columns)

    portfolio = portfolio.div(portfolio.abs().sum(axis=1), axis=0)
    portfolio = portfolio.fillna(0).loc[start_date:end_date]

    # Turn off the inner backtest plots/prints so it doesn't spam your console during the grid search
    net_sr, gross_sr, turnover, pnl = backtest_portfolio(
        portfolio,
        returns.loc[start_date:end_date],
        universe.loc[start_date:end_date],
        plot_=False,  # Set to False for the loop
        print_=False, # Set to False for the loop
        title="Feature Sensitivity"
    )

    return net_sr, gross_sr, turnover

In [7]:
def plot_feature_sensitivity(initial_weights_dict, eval_fn, feature_names, sigma=0.5, N=20):
    import os
    import matplotlib.pyplot as plt
    
    # 1. Map the dictionary to an ordered numpy array
    base_weights = np.zeros(len(feature_names))
    for i, fname in enumerate(feature_names):
        # Default to 0.0 if the feature isn't explicitly in your dict
        base_weights[i] = initial_weights_dict.get(fname, 0.0)

    # Create folder in Kaggle working directory
    save_dir = "/kaggle/working/feature_sensitivity_plots"
    os.makedirs(save_dir, exist_ok=True)
    
    best_weights_found = {}
    
    for i, fname in enumerate(feature_names):
        
        # Sample candidates around the initial weight
        candidates = np.random.normal(base_weights[i], sigma, size=N)
        # Append the exact initial weight so it's always explicitly evaluated
        candidates = np.append(candidates, base_weights[i])

        net_sharpes, gross_sharpes, turnovers = [], [], []

        for c in candidates:
            w_temp = base_weights.copy()
            w_temp[i] = c
            
            # Unpack the 3 metrics
            n_sr, g_sr, to = eval_fn(w_temp)
            net_sharpes.append(n_sr)
            gross_sharpes.append(g_sr)
            turnovers.append(to)

        candidates = np.array(candidates)
        net_sharpes = np.array(net_sharpes)
        gross_sharpes = np.array(gross_sharpes)
        turnovers = np.array(turnovers)

        # Sort arrays based on candidate weights for a smooth line plot
        idx = np.argsort(candidates)
        sorted_c = candidates[idx]
        sorted_nsr = net_sharpes[idx]
        sorted_gsr = gross_sharpes[idx]
        sorted_to = turnovers[idx]

        # Find the max Net Sharpe for highlighting
        max_idx = np.argmax(sorted_nsr)
        max_weight = sorted_c[max_idx]
        max_nsr = sorted_nsr[max_idx]

        # --- Plotting ---
        fig, ax1 = plt.subplots(figsize=(8, 5))

        # Primary Axis (Left) for Sharpe Ratios
        ax1.plot(sorted_c, sorted_gsr, marker='o', color='royalblue', alpha=0.6, label='Gross Sharpe')
        ax1.plot(sorted_c, sorted_nsr, marker='s', color='forestgreen', label='Net Sharpe')
        ax1.axvline(base_weights[i], linestyle='--', color='gray', label='Initial Weight')
        
        # Highlight Max Net Sharpe
        ax1.scatter(max_weight, max_nsr, color='crimson', s=120, zorder=5, edgecolors='black', 
                    label=f'Max Net SR\n(W={max_weight:.4f}, SR={max_nsr:.4f})')

        ax1.set_xlabel("Feature Weight")
        ax1.set_ylabel("Sharpe Ratio")
        ax1.grid(True, alpha=0.3)

        # Secondary Axis (Right) for Turnover
        ax2 = ax1.twinx()
        ax2.plot(sorted_c, sorted_to, marker='^', color='darkorange', linestyle=':', label='Turnover (%)')
        ax2.set_ylabel("Turnover (%)")

        # Combine Legends from both axes
        lines_1, labels_1 = ax1.get_legend_handles_labels()
        lines_2, labels_2 = ax2.get_legend_handles_labels()
        ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper center', 
                   bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False)

        plt.title(f"Sensitivity: {fname}")
        
        # Save plot
        filename = f"{i:02d}_{fname}_init_{base_weights[i]:.4f}.png"
        filepath = os.path.join(save_dir, filename)
        plt.savefig(filepath, bbox_inches='tight')

        plt.close() # 🚀 VERY IMPORTANT (prevents memory issues)
    
    # --- Print the resulting best weights ---
    print("\n" + "="*50)
    print("🏆 BEST WEIGHTS FOUND 🏆")
    print("="*50)
    print("next_initial_weights = {")
    for f, w in best_weights_found.items():
        # Only print if it's not basically zero (to keep the dictionary clean)
        if abs(w) > 1e-4:
            print(f'    "{f}": {w:.4f},')
    print("}")
    print("="*50 + "\n")

In [8]:
# Define your targeted starting weights here
coeff_dict = {
    "macd": 0.0,
    "trend_1_3": 0.0,
    "trend_5_20": 0.0,
    "trend_20_60": 0.0,
    "aroon": 0.0,
    "trix": 0.0,
    "ichimoku": 0.0,
    "know_sure_thing": 0.0,
    "average_true_range": 0.0,
    "volatility_20": 0.0,
    "volatility_60": 0.0,
    "on_balance_volume": 0.0,
    "ease_of_movement": 0.0,
    "accumulation_distribution_index": 0.0,
    "chaikin_money_flow": 0.0,
    "relative_strength_index": 0.0,
    "stochastic_oscillator": 0.0,
    "williams_r": 0.0,
    "volume": 0.0,
    "commodity_channel_index": 0.0,
    "ultimate_oscillator": 0.0,
    "chande_momentum_oscillator": 0.0
}

plot_feature_sensitivity(
    initial_weights_dict=coeff_dict,
    eval_fn=evaluate_weights_fast,
    feature_names=feature_names,
    sigma=3, # Adjusted slightly higher since your initial weights are quite large
    N=40
)


🏆 BEST WEIGHTS FOUND 🏆
next_initial_weights = {
}



In [9]:
# TITLE = "sample_strategy"

In [10]:
# # A Benchmark Strategy for your reference: 
# # This is the code used to generate the Benchmark submission you see in the Kaggle Leaderboard

# # This strategy shows how you can combine different features 
# def generate_portfolio_vectorized(
#     entire_features: pd.DataFrame,
#     universe: pd.DataFrame,
#     start_date: str,
#     end_date: str,
# ):
#     # Validate date format
#     try:
#         start_dt = datetime.datetime.strptime(start_date, '%Y-%m-%d')
#         end_dt = datetime.datetime.strptime(end_date, '%Y-%m-%d')
#         cutoff_date = datetime.datetime.strptime('2005-01-01', '%Y-%m-%d')
#     except ValueError:
#         raise ValueError("start_date and end_date must be strings in 'YYYY-MM-DD' format.")

#     # Ensure start_date is before end_date
#     if start_dt >= end_dt:
#         raise ValueError("start_date must be earlier than end_date.")

#     # Ensure start_date is not before '2005-01-01'
#     if start_dt < cutoff_date:
#         raise ValueError("start_date must be later than '2005-01-01'.")

#     # Get trading days within the date range
#     trading_days = universe.index[(universe.index >= start_dt) & (universe.index <= end_dt)]

#     if len(trading_days) == 0:
#         raise ValueError("No Trading Days in the specified dates")
    
#     portfolio = 0

#     universe_boolean = universe.loc[:end_date].astype(bool)
    
#     features_ = entire_features.loc[:end_date]

#     mom = process_signal(features["trend_20_60"], universe_boolean)
#     vol = process_signal(-features["volatility_20"], universe_boolean)

#     portfolio = mom + vol
#     portfolio = portfolio.div(portfolio.abs().sum(axis=1), axis=0)
    
#     return portfolio.fillna(0).loc[start_date:end_date]

### Generate your portfolio using the `generate_portfolio_vectorized` function you wrote above

In [11]:
# benchmark_portfolio_vectorized = generate_portfolio_vectorized(
#     features,
#     universe,
#     "2005-01-03",
#     "2025-02-07"
# )

### Backtest your portfolio generated using vectorized code

Note that you can backtest your portfolio till `2019-12-31` since this is the last date in the training period. You don't have access to returns after this date.

In [12]:
# sr_vectorized, pnl_vectorized = backtest_portfolio(benchmark_portfolio_vectorized.loc[:"2019"], returns.loc[:"2019"], universe.loc[:"2019"], True, True, title=TITLE)

### Benchmark Strategy: Iterative Portfolio Generation

Although the Vectorized Function generated the portfolio very quickly, it is very easy to look into the future data if you are not careful. For instance, remove the `shift(1)` operation and see the performance of the portfolio 😊. Hence, if your vectorized code has a forward bias [lookahead bias], you may see very high [or very low] sharpe ratios which may never be realised in real trading.

To avoid making these mistakes, we simulate our portfolio in a daily iterative fashion, where we call the `get_weights(features: pd.DataFrame, today_universe: pd.Series) -> dict[str, float]` function with **only** the past features data and the current day's trading universe. 

### Function Inputs:
- `features(pd.DataFrame)`:  
  - Contains various stock features indexed by date and stock identifiers.  
  - The features are structured in a **MultiIndex format**, where level 0 represents **feature names** (e.g., "macd", "volatility_60"), and level 1 represents **stock identifiers** (e.g., "1", "2", ..., "2167").  

- `today_universe(pd.Series)`:  
  - A series indicating which stocks can be traded on the current day.  
  - Contains **binary values (0 or 1)**, where **1** means a stock is **tradable**, and **0** means it is not.

You have to change this code, and write your own strategy code inside this function. Make sure it follows the same semantics as explained above.

In [13]:
# # A Benchmark Strategy for your reference: 
# # This is the code used to generate the Benchmark submission you see in the Kaggle Leaderboard

# # This strategy shows how you can combine different features 
# def get_weights(features: pd.DataFrame, today_universe: pd.Series) -> dict[str, float]:
    
#     """
#     Calculate stock weights for the portfolio on the current trading day.

#     Parameters:
#     -----------
#     features : pd.DataFrame
#         A pandas DataFrame containing feature data.
#         - Index: Datetime (chronological order).
#         - Columns: MultiIndex with two levels:
#           - Level 0: Feature names (e.g., "macd", "ichimoku", ..., "volatility_60").
#           - Level 1: Stock identifiers (e.g., "1", "2", ..., "2167").

#     today_universe : pd.Series
#         A pandas Series indicating the stock universe for the current day.
#         - Index: Stock identifiers.
#         - Values: 0 or 1, where 1 means the stock is in the universe and 
#           0 means it is not in the universe.

#     Returns:
#     --------
#     dict[str, float]
#         A dictionary where:
#         - Keys: Stock identifiers (strings).
#         - Values: Weights/positions of the stocks (floats) for the current trading day.
#     """
#     if features.shape[0] == 0:
#         return (today_universe * 1).replace(0, np.nan).dropna().fillna(0).to_dict()

#     last_day_features = features.iloc[-1].unstack().sort_index(axis=1)

#     alpha = 0

#     signal1 = last_day_features.loc["on_balance_volume"]
#     signal1 = get_universe_adjusted_series(signal1, today_universe)
#     signal1 = signal1.rank(method="min", ascending=True)
#     signal1 = scale_weights_to_one(signal1)
#     signal1 = signal1.fillna(0)

#     signal2 = last_day_features.loc["ultimate_oscillator"]
#     signal2 = get_universe_adjusted_series(signal2, today_universe)
#     signal2 = signal2.rank(method="min", ascending=True)
#     signal2 = scale_weights_to_one(signal2)
#     signal2 = signal2.fillna(0)

#     alpha = signal1 - signal2
#     alpha = scale_to_book_long_short(alpha)
#     alpha = alpha.replace(0, np.nan).dropna()

#     return alpha.to_dict()

### Generate your portfolio using the `generate_portfolio` and `get_weights` function you wrote above

Since this function is iteratively called on every trading day, it takes a lot of time [about 40 mintues] to generate the entire portfolio dataframe from `2005-01-03` to `2025-02-07`. Hence, to show an example we call it only for a one year period from `2010-01-01` to `2010-12-31`.

In [14]:
# benchmark_portfolio = generate_portfolio(
#     get_weights,
#     features,
#     universe,
#     "2010-01-01",
#     "2010-12-31",
# )

### Backtest your portfolio

Note that you can backtest your portfolio till `2019-12-31` since this is the last date in the training period. You don't have access to returns after this date.

In [15]:
# sr, pnl = backtest_portfolio(benchmark_portfolio.loc["2010"], returns.loc["2010"], universe.loc["2010"], True, True, title=TITLE)

You can also check the performance of your vectorized portfolio in this period to see if they match!

In [16]:
# sr, pnl = backtest_portfolio(benchmark_portfolio_vectorized.loc["2010"], returns.loc["2010"], universe.loc["2010"], True, True, title=TITLE)

### Comparing Iterative and Vectorized Portfolio Implementations
This function evaluates **iterative** and **vectorized** portfolio generation methods by comparing their PnL correlation. If **correlation ≥ 0.98**, both implementations are considered equivalent.
#### Steps:
1. Selects a random start date.
2. Generates portfolios using both methods.
3. Backtests portfolios and computes PnLs.
4. Validates PnL correlation.
#### Criteria:
- **Pass:** Correlation **≥ 0.98** (implementations match).
- **Fail:** Correlation **< 0.98** (error raised).
#### Inputs:
- `contestant_get_weights`: Function for portfolio weights.
- `contestant_vectorized_portfolio`: A Pandas DataFrame containing portfolio weights generated using Vectorized Implementation
- `entire_features`: Feature data (MultiIndex columns).
- `universe`: Tradable stocks (binary).
- `returns`: Daily stock returns.
#### Output:
- Prints PnL correlation or raises an error if mismatched.

In [17]:
# match_implementations(get_weights, benchmark_portfolio_vectorized, features, universe, returns)

### Final Notes
- We recommend using vectorized code to test out your strategies. This will be easier and faster to run but make sure to `shift(1)` feature dataframes in order to avoid lookahead or forward bias.
- At the end when you have decided your final strategy that you want to submit for the competition, we advise you to write code for `get_weights` which will help iteratively generate your portfolio. 
- Finally, before submitting make sure to run `match_implementations` to make sure that both versions of your code produce the same portfolio
- If these two portfolios match, you can submit the one which was produced by `generate_portfolio_vectorized` without waiting for the iterative portfolio. You don't need to run the `generate_portfolio` function for 20 years!
- We will check that the submission you made on Kaggle matches with the portfolio generated by your code. If these two don't match, you will be eliminated from the competition.

In [18]:
# # Submit this csv file on kaggle
# benchmark_portfolio_vectorized.to_csv("submission.csv")